In [1]:
# ============================================
# 02 - DATA PREPARATION
# ============================================

import os
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from PIL import Image

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print("PyTorch version:", torch.__version__)
print("Data preparation notebook initialized.")

PyTorch version: 2.9.1+cpu
Data preparation notebook initialized.


In [2]:
# ============================================
# DATASET CONFIGURATION
# ============================================

# Path to the original dataset
DATASET_PATH = Path(
    r"D:\AI_Diploma\Training tasks\Sprint_2\dataset"
)

# Path where we saved our fixed splits
SPLIT_PATH = Path("data_splits")


# ============================================
# CLASS CONFIGURATION
# ============================================

# Fixed class order for the entire project
class_names = [
    "Calculus",
    "Caries",
    "Gingivitis",
    "Hypodontia",
    "Mouth Ulcer",
    "Tooth Discoloration"
]

# Convert class name → numerical label
class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(class_names)
}

# Convert numerical label → class name
idx_to_class = {
    idx: class_name
    for class_name, idx in class_to_idx.items()
}


# ============================================
# LOAD FIXED DATASET SPLITS
# ============================================

train_df = pd.read_csv(
    SPLIT_PATH / "train.csv"
)

val_df = pd.read_csv(
    SPLIT_PATH / "validation.csv"
)

test_df = pd.read_csv(
    SPLIT_PATH / "test.csv"
)


# ============================================
# DISPLAY INFORMATION
# ============================================

print("Dataset path:")
print(DATASET_PATH)

print("\nClass mapping:")
for idx, class_name in idx_to_class.items():
    print(f"{idx} → {class_name}")

print("\nDataset splits:")
print(f"Training   : {len(train_df)}")
print(f"Validation : {len(val_df)}")
print(f"Test       : {len(test_df)}")

print("\nColumns:")
print(train_df.columns.tolist())

Dataset path:
D:\AI_Diploma\Training tasks\Sprint_2\dataset

Class mapping:
0 → Calculus
1 → Caries
2 → Gingivitis
3 → Hypodontia
4 → Mouth Ulcer
5 → Tooth Discoloration

Dataset splits:
Training   : 7184
Validation : 1540
Test       : 1540

Columns:
['file_name', 'extension', 'relative_path', 'top_level_folder', 'hash', 'class']


In [3]:
# ============================================
# PYTORCH DATASET CLASS
# ============================================

class OralDiseaseDataset(Dataset):

    def __init__(self, dataframe, dataset_path, transform=None):
        """
        dataframe:
            DataFrame containing image paths and class labels.

        dataset_path:
            Root directory of the original dataset.

        transform:
            Image transformations/augmentation.
        """

        self.dataframe = dataframe.reset_index(drop=True)
        self.dataset_path = Path(dataset_path)
        self.transform = transform

    def __len__(self):
        """
        Return the number of images in the dataset.
        """
        return len(self.dataframe)

    def __getitem__(self, index):
        """
        Load one image and its corresponding label.
        """

        # Get information about this image
        row = self.dataframe.iloc[index]

        # Construct the full image path
        image_path = self.dataset_path / row["relative_path"]

        # Load image
        image = Image.open(image_path).convert("RGB")

        # Convert class name → numerical label
        label = class_to_idx[row["class"]]

        # Apply transformations
        if self.transform:
            image = self.transform(image)

        return image, label


print("OralDiseaseDataset class created successfully.")

OralDiseaseDataset class created successfully.


In [5]:
# ============================================
# IMAGE TRANSFORMS
# ============================================

IMAGE_SIZE = 224

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Training: augmentation enabled
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.9, 1.1)
    ),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# Validation/Test: NO augmentation
val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

print("Transforms created successfully.")

Transforms created successfully.


In [6]:
train_dataset = OralDiseaseDataset(
    dataframe=train_df,
    dataset_path=DATASET_PATH,
    transform=train_transform
)

val_dataset = OralDiseaseDataset(
    dataframe=val_df,
    dataset_path=DATASET_PATH,
    transform=val_test_transform
)

test_dataset = OralDiseaseDataset(
    dataframe=test_df,
    dataset_path=DATASET_PATH,
    transform=val_test_transform
)

print("Dataset sizes:")
print("----------------")
print("Training   :", len(train_dataset))
print("Validation :", len(val_dataset))
print("Test       :", len(test_dataset))

Dataset sizes:
----------------
Training   : 7184
Validation : 1540
Test       : 1540


In [7]:
# ============================================
# INSPECT ONE TRAINING SAMPLE
# ============================================

# Select the first image from the training dataset
image_tensor, label = train_dataset[0]

# Display basic information about the sample
print("Image tensor shape:", image_tensor.shape)
print("Image tensor data type:", image_tensor.dtype)
print("Numerical label:", label)
print("Class name:", idx_to_class[label])

# Display the original image information from the DataFrame
sample_row = train_df.iloc[0]

print("\nOriginal image information:")
print("File name:", sample_row["file_name"])
print("Class:", sample_row["class"])
print("Relative path:", sample_row["relative_path"])

Image tensor shape: torch.Size([3, 224, 224])
Image tensor data type: torch.float32
Numerical label: 3
Class name: Hypodontia

Original image information:
File name: (647).JPG
Class: Hypodontia
Relative path: hypodontia\hypodontia\(647).JPG


In [8]:
# ============================================
# CREATE DATALOADERS
# ============================================

# Number of images processed together in one batch
BATCH_SIZE = 32

# Number of worker processes used to load images.
# 0 is safest on Windows and avoids multiprocessing issues.
NUM_WORKERS = 0


# ============================================
# TRAINING DATALOADER
# ============================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,          # Randomize training order every epoch
    num_workers=NUM_WORKERS
)


# ============================================
# VALIDATION DATALOADER
# ============================================

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,         # Keep validation deterministic
    num_workers=NUM_WORKERS
)


# ============================================
# TEST DATALOADER
# ============================================

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,         # Keep test evaluation deterministic
    num_workers=NUM_WORKERS
)


# ============================================
# DISPLAY DATALOADER INFORMATION
# ============================================

print("DataLoaders created successfully.")
print("----------------------------------")

print("Batch size:", BATCH_SIZE)

print("Training batches   :", len(train_loader))
print("Validation batches :", len(val_loader))
print("Test batches       :", len(test_loader))

DataLoaders created successfully.
----------------------------------
Batch size: 32
Training batches   : 225
Validation batches : 49
Test batches       : 49


In [9]:
# ============================================
# INSPECT ONE BATCH
# ============================================

# Get the first batch from the training DataLoader
images, labels = next(iter(train_loader))

# Display the batch information
print("Batch inspection")
print("----------------")

# Shape should be:
# [batch_size, channels, height, width]
print("Images shape:", images.shape)

# Labels should contain one class index per image
print("Labels shape:", labels.shape)

# Show the data types
print("Images data type:", images.dtype)
print("Labels data type:", labels.dtype)

# Display the labels of the first batch
print("\nBatch labels:")
print(labels)

# Convert numerical labels back to class names
print("\nBatch classes:")

for label in labels:
    print(idx_to_class[label.item()])

Batch inspection
----------------
Images shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Images data type: torch.float32
Labels data type: torch.int64

Batch labels:
tensor([1, 5, 4, 4, 5, 5, 1, 3, 1, 1, 3, 4, 5, 3, 4, 5, 5, 5, 4, 1, 1, 5, 4, 1,
        4, 1, 4, 4, 5, 3, 2, 3])

Batch classes:
Caries
Tooth Discoloration
Mouth Ulcer
Mouth Ulcer
Tooth Discoloration
Tooth Discoloration
Caries
Hypodontia
Caries
Caries
Hypodontia
Mouth Ulcer
Tooth Discoloration
Hypodontia
Mouth Ulcer
Tooth Discoloration
Tooth Discoloration
Tooth Discoloration
Mouth Ulcer
Caries
Caries
Tooth Discoloration
Mouth Ulcer
Caries
Mouth Ulcer
Caries
Mouth Ulcer
Mouth Ulcer
Tooth Discoloration
Hypodontia
Gingivitis
Hypodontia
